In [1]:
import polars as pl
import os

In [2]:
data_path = "preprocessing/clean_data"

items = pl.scan_parquet(os.path.join(data_path, "items_cleaned.parquet"))

purchases = pl.scan_parquet(os.path.join(data_path, "purchases_cleaned.parquet"))

users = pl.scan_parquet(os.path.join(data_path, "users_cleaned.parquet"))

In [3]:
# 1. Tính toán hành vi mua sắm từ bảng purchases
reference_date = pl.lit("2025-01-01 00:00:00").str.to_datetime()

user_behavior = (
    purchases
    .group_by("customer_id")
    .agg([
        pl.len().alias("u_total_transactions"),
        pl.col("quantity").sum().alias("u_total_quantity"),
        (pl.col("price") * pl.col("quantity") - pl.col("discount")).sum().alias("u_total_spend"),
        (pl.col("discount").sum() / (pl.col("price") * pl.col("quantity")).sum()).alias("u_discount_ratio"),
        ((reference_date - pl.col("created_date").max()).dt.total_days()).alias("u_recency_days"),
        pl.col("item_id").n_unique().alias("u_unique_items_bought")
    ])
    .with_columns(
        u_avg_order_value=pl.col("u_total_spend") / pl.col("u_total_transactions")
    )
)

# 2. Kết hợp với thông tin cá nhân từ bảng users
user_features = (
    users
    .select([
        "customer_id",
        pl.col("gender").alias("u_gender"),
        pl.col("province").alias("u_province"),
        pl.col("region").alias("u_region"),
        pl.col("membership").alias("u_membership"),
        ((reference_date - pl.col("created_date")).dt.total_days()).alias("u_account_age_days")
    ])
    .join(user_behavior, on="customer_id", how="inner")
)

In [4]:
# 1. Thống kê độ hot & sức bán từ purchases
item_stats = (
    purchases
    .group_by("item_id")
    .agg([
        pl.len().alias("i_total_orders"),
        pl.col("quantity").sum().alias("i_total_units_sold"),
        pl.col("customer_id").n_unique().alias("i_unique_buyers")
    ])
)

# 2. Kết hợp với đặc tính sản phẩm từ items
item_features = (
    items
    .select([
        "item_id",
        pl.col("category_l1").alias("i_category_l1"),
        pl.col("category_l2").alias("i_category_l2"),
        pl.col("brand").alias("i_brand"),
        pl.col("price").cast(pl.Float64).alias("i_price"),
        pl.col("item_type").alias("i_item_type"),
        pl.col("age_group").alias("i_age_group"),
        pl.col("gender_target").alias("i_gender_target")
    ])
    .join(item_stats, on="item_id", how="inner")
)

In [5]:
user_item_interactions = (
    purchases
    .group_by(["customer_id", "item_id"])
    .agg([
        pl.len().alias("ui_past_purchase_count"),
        pl.col("quantity").sum().alias("ui_past_quantity"),
        ((reference_date - pl.col("created_date").max()).dt.total_days()).alias("ui_days_since_last_buy")
    ])
)

In [6]:
# Ghép các đặc trưng vào từng cặp (User, Item)
full_features = (
    user_item_interactions
    .join(user_features, on="customer_id", how="left")
    .join(item_features, on="item_id", how="left")
)

full_features.limit(5).collect()

customer_id,item_id,ui_past_purchase_count,ui_past_quantity,ui_days_since_last_buy,u_gender,u_province,u_region,u_membership,u_account_age_days,u_total_transactions,u_total_quantity,u_total_spend,u_discount_ratio,u_recency_days,u_unique_items_bought,u_avg_order_value,i_category_l1,i_category_l2,i_brand,i_price,i_item_type,i_age_group,i_gender_target,i_total_orders,i_total_units_sold,i_unique_buyers
i32,str,u32,i32,i64,str,str,str,str,i64,u32,i32,"decimal[38,4]","decimal[38,4]",i64,u32,"decimal[38,4]",str,str,str,f64,str,str,str,u32,i32,u32
464836,"""4603024000001""",1,1,364,"""Nữ""","""Tây Ninh""","""Đông Nam Bộ""","""Standard""",3365,12,12,1088200.0000,0.0081,32,5,90683.3333,"""Hóa mỹ phẩm cho bé""","""Nước rửa bình sữa""","""Animo""",89000.0,"""Nước rửa bình sữa""","""Từ 0M""","""Unisex""",103908,110476,81466
2785992,"""0175000000007""",1,1,274,"""Nữ""","""Hồ Chí Minh""","""Đông Nam Bộ""","""Standard""",1870,4,5,1569000.0000,0.0000,57,4,392250.0000,"""Textile""","""Khăn em bé""","""Animo""",79000.0,"""Khăn gạc, khăn sữa""","""Từ 0M""","""Unisex""",82198,102455,69627
7021360,"""4848000000001""",1,1,120,"""Nữ""","""Bình Thuận""","""Duyên hải Nam Trung Bộ""","""Diamond""",394,146,164,16065584.0000,0.1143,9,125,110038.2466,"""Hóa mỹ phẩm gia đình""","""Chăm sóc tóc""","""Dove""",155000.0,"""Dầu xả""","""Gia đình""","""Gia đình""",2164,2209,2054
7940856,"""3533000000230""",1,1,83,"""Nữ""","""Bình Định""","""Duyên hải Nam Trung Bộ""","""Gold""",83,13,15,1781459.9998,0.2552,83,12,137035.3846,"""Thời trang""","""Quần áo & Phụ kiện sơ sinh""","""Không xác định""",119000.0,"""Quần""","""Từ 0M""","""Sơ sinh""",1299,1335,1253
5231055,"""7037000000003""",2,2,111,"""Nữ""","""Bà Rịa - Vũng Tàu""","""Đông Nam Bộ""","""Standard""",955,17,23,2142000.0000,0.0553,111,14,126000.0000,"""Hóa mỹ phẩm cho bé""","""Vệ sinh cho bé""","""Vệ sinh, chăm sóc da Aga-ae""",259000.0,"""Sữa tắm gội 2in1""","""Từ 0M""","""Unisex""",4867,4903,4440


In [7]:
full_features.collect().shape

(24700599, 27)

In [8]:
import numpy as np

SPLIT_DATE = pl.lit("2024-11-01 00:00:00").str.to_datetime()

purchases_history = purchases.filter(pl.col("created_date") < SPLIT_DATE)

purchases_future = purchases.filter(pl.col("created_date") >= SPLIT_DATE)

In [9]:
# 1. Tính toán hành vi RFM, chi tiêu và giá mua trung bình từ lịch sử
user_behavior = (
    purchases_history
    .group_by("customer_id")
    .agg([
        pl.len().alias("u_past_orders_count"),
        pl.col("quantity").sum().alias("u_past_total_quantity"),
        (pl.col("price") * pl.col("quantity") - pl.col("discount")).sum().alias("u_past_total_spend"),
        (pl.col("discount").sum() / (pl.col("price") * pl.col("quantity")).sum()).alias("u_discount_ratio"),
        ((SPLIT_DATE - pl.col("created_date").max()).dt.total_days()).alias("u_days_since_last_order"),
        pl.col("item_id").n_unique().alias("u_unique_items_bought"),
        pl.col("price").cast(pl.Float64).mean().alias("u_avg_item_price")  # Giá trung bình mỗi món khách hay mua
    ])
    .with_columns(
        u_avg_order_value=pl.col("u_past_total_spend") / pl.col("u_past_orders_count")
    )
)

# 2. Kết hợp với nhân khẩu học từ bảng users
user_features = (
    users
    .select([
        "customer_id",
        pl.col("gender").alias("u_gender"),
        pl.col("province").alias("u_province"),
        pl.col("region").alias("u_region"),
        pl.col("membership").alias("u_membership"),
        ((SPLIT_DATE - pl.col("created_date")).dt.total_days()).alias("u_account_age_days")
    ])
    .join(user_behavior, on="customer_id", how="left")
)

In [10]:
# 1. Thống kê độ phổ biến của item trong 10 tháng đầu năm
item_stats = (
    purchases_history
    .group_by("item_id")
    .agg([
        pl.len().alias("i_past_orders_count"),
        pl.col("quantity").sum().alias("i_past_units_sold"),
        pl.col("customer_id").n_unique().alias("i_past_unique_buyers")
    ])
)

# 2. Kết hợp với thuộc tính sản phẩm từ items
item_features = (
    items
    .select([
        "item_id",
        pl.col("category_l1").alias("i_category_l1"),
        pl.col("category_l2").alias("i_category_l2"),
        pl.col("brand").alias("i_brand"),
        pl.col("price").cast(pl.Float64).alias("i_price"),
        pl.col("item_type").alias("i_item_type"),
        pl.col("age_group").alias("i_age_group"),
        pl.col("gender_target").alias("i_gender_target")
    ])
    .join(item_stats, on="item_id", how="left")
)

In [11]:
# 1. Kết hợp item metadata vào purchases_history để tính mức độ gắn bó của User với Category / Brand
purchases_with_meta = purchases_history.join(
    items.select(["item_id", "category_l1", "category_l2", "brand"]),
    on="item_id",
    how="left"
)
# 2. Số lần user mua Category L1
u_cat_l1_affinity = (
    purchases_with_meta
    .filter(pl.col("category_l1").is_not_null())
    .group_by(["customer_id", "category_l1"])
    .agg(pl.len().alias("u_cat_l1_buy_count"))
)
# 3. Số lần user mua Category L2
u_cat_l2_affinity = (
    purchases_with_meta
    .filter(pl.col("category_l2").is_not_null())
    .group_by(["customer_id", "category_l2"])
    .agg(pl.len().alias("u_cat_l2_buy_count"))
)
# 4. Số lần user mua Brand
u_brand_affinity = (
    purchases_with_meta
    .filter(pl.col("brand").is_not_null())
    .group_by(["customer_id", "brand"])
    .agg(pl.len().alias("u_brand_buy_count"))
)
# 5. Tương tác trực tiếp User - Item trong quá khứ
user_item_history = (
    purchases_history
    .group_by(["customer_id", "item_id"])
    .agg([
        pl.len().alias("ui_past_purchase_count"),
        pl.col("quantity").sum().alias("ui_past_quantity"),
        ((SPLIT_DATE - pl.col("created_date").max()).dt.total_days()).alias("ui_days_since_last_buy")
    ])
)

# 6. ĐẶC TRƯNG MUA KÈM (BASKET CO-OCCURRENCE):
basket_items = purchases_history.select(["customer_id", "created_date", "item_id"]).unique()
cooccur_df = (
    basket_items
    .join(basket_items, on=["customer_id", "created_date"])
    .filter(pl.col("item_id") != pl.col("item_id_right"))
    .group_by(["item_id", "item_id_right"])
    .agg(pl.len().alias("cooccur_count"))
    .filter(pl.col("cooccur_count") >= 10)
)
os.makedirs("train_data", exist_ok=True)
cooccur_df.sink_parquet("train_data/item_cooccurrence.parquet", compression="zstd")

top_companion = (
    pl.read_parquet("train_data/item_cooccurrence.parquet")
    .sort(["item_id", "cooccur_count"], descending=[False, True])
    .group_by("item_id")
    .first()
    .rename({
        "item_id_right": "companion_item_id", 
        "cooccur_count": "companion_cooccur_count"
    })
)

user_past_items = purchases_history.select(["customer_id", "item_id"]).unique()
companion_affinity = (
    top_companion.lazy()
    .join(user_past_items, left_on="companion_item_id", right_on="item_id", how="inner")
    .select([
        "customer_id",
        "item_id",
        pl.lit(1, dtype=pl.Int8).alias("ui_bought_companion"),
        pl.col("companion_cooccur_count").alias("ui_companion_cooccur_count")
    ])
)


In [12]:
# 1. Mẫu Dương: Các cặp (customer_id, item_id) thực tế mua trong 2 tháng cuối năm
pos_samples = (
    purchases_future
    .select(["customer_id", "item_id"])
    .unique()
    .collect()
    .with_columns(
        customer_id=pl.col("customer_id").cast(pl.Int32),
        label=pl.lit(1, dtype=pl.Int8)
    )
)

# 2. Sinh Mẫu Âm tỷ lệ 1 : 2
all_items_array = items.select("item_id").unique().collect()["item_id"].to_numpy()
n_all_items = len(all_items_array)

NEG_RATIO = 2
neg_users = np.repeat(pos_samples["customer_id"].to_numpy(), NEG_RATIO)
rand_indices = np.random.randint(0, n_all_items, size=len(neg_users))
neg_items = all_items_array[rand_indices]

neg_samples = pl.DataFrame({
    "customer_id": neg_users.astype(np.int32),
    "item_id": neg_items,
    "label": np.zeros(len(neg_users), dtype=np.int8)
})

labeled_pairs = pl.concat([pos_samples, neg_samples])
print(f"Tổng số mẫu: {len(labeled_pairs):,} (Dương: {len(pos_samples):,}, Âm: {len(neg_samples):,})")

Tổng số mẫu: 15,137,742 (Dương: 5,045,914, Âm: 10,091,828)


In [13]:
final_train_df = (
    labeled_pairs.lazy()
    .join(item_features, on="item_id", how="left")
    .join(user_features, on="customer_id", how="left")
    .join(user_item_history, on=["customer_id", "item_id"], how="left")
    .join(u_cat_l1_affinity, left_on=["customer_id", "i_category_l1"], right_on=["customer_id", "category_l1"], how="left")
    .join(u_cat_l2_affinity, left_on=["customer_id", "i_category_l2"], right_on=["customer_id", "category_l2"], how="left")
    .join(u_brand_affinity, left_on=["customer_id", "i_brand"], right_on=["customer_id", "brand"], how="left")
    # THÊM DÒNG NÀY ĐỂ GHÉP ĐẶC TRƯNG MUA KÈM:
    .join(companion_affinity, on=["customer_id", "item_id"], how="left")
    .with_columns([
        # Điền null cho 2 cột mua kèm
        pl.col("ui_bought_companion").fill_null(0),
        pl.col("ui_companion_cooccur_count").fill_null(0),
        # ... các cột cũ giữ nguyên
        pl.col("ui_past_purchase_count").fill_null(0),
        pl.col("ui_past_quantity").fill_null(0),
        pl.col("ui_days_since_last_buy").fill_null(999),
        pl.col("u_cat_l1_buy_count").fill_null(0),
        pl.col("u_cat_l2_buy_count").fill_null(0),
        pl.col("u_brand_buy_count").fill_null(0),
        pl.col("u_past_orders_count").fill_null(0),
        pl.col("u_past_total_quantity").fill_null(0),
        pl.col("u_past_total_spend").fill_null(0),
        pl.col("u_discount_ratio").fill_null(0.0),
        pl.col("u_days_since_last_order").fill_null(999),
        pl.col("u_unique_items_bought").fill_null(0),
        pl.col("u_avg_item_price").fill_null(0.0),
        pl.col("u_avg_order_value").fill_null(0.0),
        pl.col("i_past_orders_count").fill_null(0),
        pl.col("i_past_units_sold").fill_null(0),
        pl.col("i_past_unique_buyers").fill_null(0)
    ])
    .with_columns([
        (pl.col("i_price") / (pl.col("u_avg_item_price") + 1.0)).alias("ui_price_ratio_user_avg"),
        (pl.col("i_price") - pl.col("u_avg_item_price")).alias("ui_price_diff_user_avg"),
        (pl.col("u_cat_l1_buy_count") / (pl.col("u_past_orders_count") + 1.0)).alias("ui_cat_l1_affinity_ratio"),
        (pl.col("u_brand_buy_count") / (pl.col("u_past_orders_count") + 1.0)).alias("ui_brand_affinity_ratio")
    ])
)

In [14]:
train_dir = "train_data"
os.makedirs(train_dir, exist_ok=True)
train_output_path = os.path.join(train_dir, "train_ranking_data.parquet")

final_train_df.sink_parquet(train_output_path, compression="zstd")